# Cálculo de emissões por atividade produtiva

Esse notebook faz o cálculo das emissões por atividade produtiva.

## 1. Inicialização

Carrega dependências e executa configurações iniciais.

In [98]:
import importlib

import numpy as np
import pandas as pd
import seaborn as sns

from redes import dados, emissoes, mip, modelo

importlib.reload(mip)
importlib.reload(modelo)

sns.set_theme(context="notebook", style="whitegrid")

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 70)

## 2. Carregamento dos dados de entrada

Carrega os dados de entrada:
- Os dados de entrada estão armazenados na pasta 'raw'. 
- O arquivo manifesto.csv contém o índice de arquivos canônicos e seus respectivos hashes. 
- O hash é verificado a cada carregamento para garantir a integridade dos dados de entrada.
- Os coeficientes de emissão estão em Gg de CO₂ por R$ milhão de produção bruta.


### 2.1. MIP brasileira de 2015

Carrega a MIP brasileira de 2015, nível 67.

Fonte: IBGE.

In [99]:
MIP = "mip_ibge_2015_67"

tabelas = dados.carregar_matriz_67(MIP)
# Preview da matriz dos coeficientes tecnicos intesetoriais (Tabela 14)
mip.carregar_coeficientes_tecnicos_67(tabelas).head().iloc[:, :5]

SHA-256 verificado: Matriz_de_Insumo_Produto_2015_Nivel_67.xls


atividade_destino,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,0.021056,0.027488,0.004699,0.000106,0.000032
0192,0.002413,0.032509,0.003979,0.000254,0.000086
0280,0.002699,0.007325,0.048358,0.000118,0.000007
0580,0.000514,0.002551,0.000358,0.015136,0.002867
0680,0.000020,0.000029,0.000010,0.000062,0.070423


Seleciona as demandas finais por produto e os coeficientes de insumos importados usados na conta de consumo.

In [100]:
# Tabelas 03 e 04 - Demanda final doméstica por produto.
# Soma consumo do governo, ISFLSF (Instituições Sem Fins Lucrativos a Serviço
# das Famílias), famílias, formação bruta de capital fixo e variação de estoques.
# Tabela 03: produção nacional; Tabela 04: produtos importados.
# Range: BU6:BY132 (iloc[5:132, 72:77]); 127 produtos × 5 componentes.
# Resultado: vetores com 127 produtos, em R$ milhões de 2015.
y_dom_produtos = tabelas["03"].iloc[5:132, 72:77].apply(pd.to_numeric, errors="raise").sum(axis=1)
y_imp_produtos = tabelas["04"].iloc[5:132, 72:77].apply(pd.to_numeric, errors="raise").sum(axis=1)
y_dom_produtos.index = pd.Index(tabelas["03"].iloc[5:132, 0].astype(str), name="produto")
y_imp_produtos.index = pd.Index(tabelas["04"].iloc[5:132, 0].astype(str), name="produto")

# Tabela 03 - Exportações de produtos nacionais.
# Range: BT6:BT132 (iloc[5:132, 71]); vetor com 127 produtos.
# Unidade: R$ milhões de 2015.
y_exp_produtos = pd.Series(
    pd.to_numeric(tabelas["03"].iloc[5:132, 71], errors="raise").to_numpy(),
    index=y_dom_produtos.index,
)

# Tabela 12 - Bm: coeficientes de insumos importados por produto e atividade.
# Range: C6:BQ132 (iloc[5:132, 2:69]).
# Linhas: 127 produtos; colunas: 67 atividades compradoras.
# Unidade: R$ de insumo importado por R$ de produção brasileira.
Bm = mip.carregar_matriz_bm_67(tabelas)


### 2.2. Intensidades de CO₂ do Brasil

Carrega as intensidades de 2011 e 2018 e estima 2015 por interpolação linear.

Unidade: Gg de CO₂ por R$ milhão.

Fonte dos valores de 2011 e 2018: Sanguinet e Azzoni (2024).


In [101]:
# Os coeficientes publicados têm duas casas decimais; a interpolação não
# aumenta a precisão dos dados originais, mesmo produzindo casas adicionais.
# Hipótese monetária: adota-se fator de conversão 1 entre a base de preços
# de 2018 dos coeficientes e os valores correntes de 2015 da MIP.
# Essa aproximação ignora a variação de preços; não é uma deflação efetiva
# e limita a interpretação dos níveis absolutos de emissões.
coeficientes_2011 = dados.carregar_coeficientes_co2("sanguinet_azzoni_2011")
coeficientes_2018 = dados.carregar_coeficientes_co2("sanguinet_azzoni_2018")

# Interpolação linear: 2015 = 2011 + 4/7 da variação entre 2011-18
assert coeficientes_2011.index.equals(coeficientes_2018.index)
peso_2015 = (2015 - 2011) / (2018 - 2011)
coeficientes_2015 = coeficientes_2011[2011] + peso_2015 * (coeficientes_2018[2018] - coeficientes_2011[2011])

matriz_coeficientes_co2 = coeficientes_2015.to_frame(name=2015)
matriz_coeficientes_co2.columns.name = "ano"
matriz_coeficientes_co2.head()

SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2011.csv
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2018.csv


ano,2015
setor_id,
S1,0.034286
S2,0.044286
S3,0.178571
S4,0.075714
S5,0.010000


### 2.3. Parâmetros do exterior representativo

Carrega as intensidades de emissões e a inversa de Leontief do exterior representativo.

Assume-se que o exterior possui as mesmas intensidades de emissões e os mesmos encadeamentos internos de produção que o Brasil (`L_ext = L`). Como `L` considera apenas insumos nacionais, essa aproximação reproduz a parcela interna da tecnologia brasileira, não seus requisitos totais de insumos. Não são modeladas as compras adicionais ao Brasil ou a outros países necessárias à produção externa; portanto, a conta de consumo não cobre integralmente as cadeias mundiais. A hipótese monetária de 2.2 também se aplica às intensidades externas.

Fonte: ver fontes em 2.1 e 2.2.


In [102]:
COEFICIENTES_EXTERIOR = "coeficientes_co2_exterior_proxy_brasil"
TECNOLOGIA_EXTERIOR = "inversa_leontief_exterior_proxy_brasil_2015"

coeficientes_exterior = dados.carregar_intensidades_co2_exterior(COEFICIENTES_EXTERIOR)
# Aplica a mesma interpolação ao cenário externo para usar 2015 em todas as contas.
intensidade_exterior_2015 = coeficientes_exterior[2011] + peso_2015 * (coeficientes_exterior[2018] - coeficientes_exterior[2011])
intensidades_co2_exterior = intensidade_exterior_2015.to_frame(name=2015)
intensidades_co2_exterior.columns.name = "ano"
L_ext = dados.carregar_inversa_leontief_exterior(TECNOLOGIA_EXTERIOR)

# Assume a mesma ordem dos setores S1–S67 e das atividades da MIP.
atividades = mip.carregar_matriz_producao_tabela_01_67(tabelas).columns
intensidades_co2 = matriz_coeficientes_co2.copy()
intensidades_co2.index = atividades
intensidades_co2.index.name = "atividade"
assert intensidades_co2_exterior.index.equals(atividades)
assert L_ext.index.equals(atividades)
assert set(intensidades_co2.columns).issubset(intensidades_co2_exterior.columns)

display(intensidades_co2_exterior.head())
display(L_ext.head().iloc[:, :5])

SHA-256 verificado: coeficientes_co2_exterior_proxy_brasil.csv
SHA-256 verificado: inversa_leontief_exterior_proxy_brasil_2015.csv


ano,2015
atividade,
0191,0.034286
0192,0.044286
0280,0.178571
0580,0.075714
0680,0.010000


atividade_destino,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,1.027732,0.054436,0.013645,0.007712,0.004759
0192,0.003118,1.040622,0.005558,0.000849,0.000472
0280,0.003578,0.009131,1.051213,0.000648,0.000400
0580,0.003023,0.004454,0.000884,1.016587,0.003983
0680,0.022982,0.019166,0.010885,0.030039,1.088226


## 3. Matriz de insumo-produto e modelos de Leontief e Ghosh

Organiza os dados da MIP por atividade e calcula os modelos de Leontief e Ghosh.


### 3.1. Dados por produto e atividade

Seleciona os dados por produto e atividade

In [103]:
# Tabela 01 - Recursos de bens e serviços - 2015 (bloco de produção).
# Mostra quais produtos (linhas) foram PRODUZIDOS por cada atividade (coluna)
# Range: H6:BV132 (iloc[5:132, 7:74]).
# Linhas: 127 produtos; colunas: 67 atividades produtoras.
V = mip.carregar_matriz_producao_tabela_01_67(tabelas)

# Tabela 03 - Oferta e demanda da produção nacional a preço básico - 2015
# Mostra quais produtos (linhas) foram CONSUMIDOS por cada atividade (coluna)
# Range: D6:BR132 (iloc[5:132, 3:70]).
# Linhas: 127 produtos; colunas: 67 atividades compradoras.
U = mip.carregar_matriz_uso_nacional_tabela_03_67(tabelas)

# Tabela 03 - Oferta e demanda da produção nacional a preço básico - 2015
# Mostra a demanda final de cada produto
# Range: BZ6:BZ132 (iloc[5:132, 77]); 
# Vetor com 127 produtos.
y_produtos = pd.Series(
    pd.to_numeric(tabelas["03"].iloc[5:132, 77], errors="raise").to_numpy(),
    index=U.index, name="demanda_final_por_produto",
)

# Tabela 13 - Participação setorial na produção dos produtos nacionais.
# Calculada através da matriz V
# q = V.sum(axis=1)         # Produção total de cada produto.
# D = V.div(q, axis=0).T    # Participação de cada atividade nessa produção.
# Range: C6:DY72 (iloc[5:72, 2:129]).
# Linhas: 67 atividades produtoras; colunas: 127 produtos.
D = mip.carregar_matriz_participacao_tabela_13_67(tabelas)

# Confirma dimensões
assert D.columns.equals(U.index) and V.columns.equals(U.columns)
assert D.index.equals(U.columns)
print(f"U — Consumo intermediário nacional: {U.shape[0]} produtos × {U.shape[1]} atividades compradoras")
print(f"V — Produção nacional: {V.shape[0]} produtos × {V.shape[1]} atividades produtoras")
print(f"D — Participação na produção: {D.shape[0]} atividades produtoras × {D.shape[1]} produtos")
print(f"y_produtos — Demanda final nacional: vetor coluna com {y_produtos.size} produtos")

U — Consumo intermediário nacional: 127 produtos × 67 atividades compradoras
V — Produção nacional: 127 produtos × 67 atividades produtoras
D — Participação na produção: 67 atividades produtoras × 127 produtos
y_produtos — Demanda final nacional: vetor coluna com 127 produtos


### 3.2. Produção, transações e demanda final

Calcula a produção bruta, as transações entre atividades e a demanda final por atividade, em R$ milhões de 2015.

In [104]:
# x - Produção bruta por atividade.
# Mostra o valor total PRODUZIDO por cada atividade, somando a coluna de V.
# Origem: V (Tabela 01, H6:BV132).
# Vetor com 67 atividades produtoras.
x = V.sum(axis=0)

# Z - Transações intermediárias nacionais entre atividades.
# Mostra quanto cada atividade forneceu como INSUMO para cada atividade compradora.
# Obtida multiplicando D (Tabela 13, 67x127), participação de cada atividade na produção de cada produto,
# por U (Tabela 03, 127x67), valor dos produtos usados como insumos por cada atividade compradora.
# Como não sabemos de qual atividade cada comprador adquiriu o produto,
# distribuímos a compra conforme a participação de cada atividade na produção nacional desse produto.
# Linhas: 67 atividades fornecedoras; colunas: 67 atividades compradoras.
Z = D @ U

# y - Demanda final por atividade produtora.
# Mostra a demanda final dos produtos atribuída às atividades que os produziram.
# Calculada com D 67x127 (Tabela 13) e y_produtos 127x1 (Tabela 03).
# Vetor com 67 atividades produtoras; inclui exportações e exclui produtos importados.
y = D @ y_produtos

# Confirma produção positiva para a divisão por x no cálculo dos coeficientes.
assert (x > 0).all()

# Balanço por atividade: produção bruta = vendas intermediárias + demanda final.
# A soma de cada linha de Z representa as vendas de insumos da atividade.
balanco = pd.DataFrame({
    "vendas_intermediarias": Z.sum(axis=1),
    "demanda_final_y": y,
    "producao_bruta_x": x,
})
# Confirma o fechamento do balanço, admitindo diferença numérica inferior a R$ 1.
erro_balanco = (x - Z.sum(axis=1) - y).abs().max()
assert erro_balanco < 1e-6, "As vendas não fecham com a produção bruta."
print(f"Matriz Z: Linhas: 67 atividades fornecedoras; colunas: 67 atividades compradoras.")
display(Z.iloc[:5, :5])
print(f"Balanço (R$ milhões): produção bruta - demanda final - consumo intermediário = {erro_balanco:.2e}")
display(balanco.head())

Matriz Z: Linhas: 67 atividades fornecedoras; colunas: 67 atividades compradoras.


atividade,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,6512.791125,3766.341671,152.308653,2.082552,5.588521
0192,746.364271,4454.324929,128.962890,5.014442,14.776775
0280,834.747542,1003.603478,1567.336903,2.326244,1.155096
0580,159.093970,349.513581,11.603577,298.687343,492.992626
0680,6.317105,4.026470,0.311644,1.224293,12111.644666


Balanço (R$ milhões): produção bruta - demanda final - consumo intermediário = 1.16e-10


,vendas_intermediarias,demanda_final_y,producao_bruta_x
0191,142986.313251,166314.686749,309301.0
0192,95443.719009,41574.280991,137018.0
0280,14343.782343,18067.217657,32411.0
0580,17291.292966,2441.707034,19733.0
0680,114123.123823,57860.876177,171984.0


### 3.3. Coeficientes técnicos

Calcula os insumos nacionais necessários por R$ 1 de produção de cada atividade e verifica o balanço entre produção, consumo intermediário e demanda final.

In [105]:
# A - Coeficientes técnicos de insumos nacionais.
# Mostra quanto de insumo da atividade fornecedora i é utilizado por R$ 1
# de produção da atividade compradora j: a_ij = z_ij / x_j.
# Divide cada coluna de Z pela produção bruta TOTAL da atividade compradora (X),
# e não apenas pelo subtotal de suas compras de insumos nacionais.
# Linhas: 67 atividades fornecedoras; colunas: 67 atividades compradoras.
# Unidade: R$ de insumo nacional por R$ de produção (coeficiente adimensional).
# Importações e impostos não entram no numerador; a soma da coluna não precisa ser 1.
A = Z.div(x, axis=1)

# A @ x recupera as vendas intermediárias nacionais de cada atividade.
# Confirma o balanço x = A @ x + y, com diferença inferior a R$ 1.
assert (x - A @ x - y).abs().max() < 1e-6

display(A.iloc[:5, :5])

# Confronta A com os coeficientes técnicos publicados pelo IBGE.
A_ibge = mip.carregar_coeficientes_tecnicos_67(tabelas)
erro_a = (A - A_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 14 do IBGE: {erro_a:.2e}")
assert erro_a < 1e-10, "A matriz A calculada diverge da Tabela 14 do IBGE."

atividade,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,0.021056,0.027488,0.004699,0.000106,0.000032
0192,0.002413,0.032509,0.003979,0.000254,0.000086
0280,0.002699,0.007325,0.048358,0.000118,0.000007
0580,0.000514,0.002551,0.000358,0.015136,0.002867
0680,0.000020,0.000029,0.000010,0.000062,0.070423


Erro absoluto máximo em relação à Tabela 14 do IBGE: 1.11e-16


### 3.4. Inversa de Leontief

Calcula a produção nacional necessária para atender à demanda final, considerando os insumos diretos e indiretos: `L = (I − A)⁻¹`.

In [106]:
# I - Matriz identidade: 1 na diagonal e 0 nas demais posições.
# Linhas e colunas: as mesmas 67 atividades de A, na mesma ordem.
I = pd.DataFrame(np.eye(len(A)), index=A.index, columns=A.columns)

# L - Inversa de Leontief (requerimentos diretos e indiretos de produção).
# Cada L[i, j] mostra a produção da atividade i necessária para atender
# a R$ 1 de demanda final pela produção da atividade j.
# Calculada com A, obtida em 3.3: L = (I - A)⁻¹.
# Linhas: 67 atividades produtoras; colunas: 67 atividades demandadas.
# Unidade: R$ de produção nacional por R$ de demanda final (adimensional).
L = pd.DataFrame(np.linalg.inv(I - A), index=A.index, columns=A.columns)

# Confirma que a demanda final observada reproduz a produção bruta de 3.2.
assert (x - L @ y).abs().max() < 1e-6

display(L.head())

# Confere se o cálculo de L reproduz os coeficientes publicados pelo IBGE
# Tabela 15 - Matriz de Leontief publicada pelo IBGE.
# Range: C6:BQ72 (iloc[5:72, 2:69]); 67 atividades × 67 atividades.
L_ibge = mip.carregar_inversa_leontief_ibge_67(tabelas)
erro_l = (L - L_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 15 do IBGE: {erro_l:.2e}")
assert erro_l < 1e-10, "A inversa calculada diverge da matriz publicada pelo IBGE."

atividade,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,1.027732,0.054436,0.013645,0.007712,0.004759,0.004920,...,0.002084,0.005709,0.004749,0.002088,0.008247,0.0
0192,0.003118,1.040622,0.005558,0.000849,0.000472,0.000402,...,0.001097,0.003206,0.002639,0.000536,0.003484,0.0
0280,0.003578,0.009131,1.051213,0.000648,0.000400,0.000482,...,0.000384,0.000683,0.000783,0.000437,0.000788,0.0
0580,0.003023,0.004454,0.000884,1.016587,0.003983,0.000898,...,0.000296,0.000731,0.000422,0.000523,0.000626,0.0
0680,0.022982,0.019166,0.010885,0.030039,1.088226,0.032999,...,0.004184,0.003928,0.003700,0.005477,0.008738,0.0


Erro absoluto máximo em relação à Tabela 15 do IBGE: 1.55e-15


### 3.5. Coeficientes e inversa de Ghosh

Calcula a parcela da produção de cada atividade destinada às demais atividades como insumo e sua propagação direta e indireta no sistema nacional: `b_ij = z_ij / x_i` e `G = (I − B)⁻¹`.

In [107]:
# B - Coeficientes de alocação de Ghosh.
# Mostra a fração da produção da fornecedora i vendida como insumo
# à compradora j: b_ij = z_ij / x_i.
# Calculada com os mesmos Z e x de 3.2.
# Divide cada LINHA de Z pela produção bruta da atividade fornecedora.
# Linhas: 67 atividades fornecedoras; colunas: 67 atividades compradoras.
# Unidade: R$ de venda intermediária nacional por R$ de produção (adimensional).
# A parcela destinada à demanda final não entra em B.
B = Z.div(x, axis=0)

print(f"Coeficientes de Gosh:")
display(B.head())

# G - Inversa de Ghosh.
# Acumula os encadeamentos de alocação da produção representados por B.
# Calculada como G = (I - B)⁻¹, usando a identidade definida no item 3.4.
# Linhas: 67 atividades de origem; colunas: 67 atividades de destino.
# Coeficientes adimensionais; a interpretação do modelo supõe B constante (não existe efeito substituição).
G = pd.DataFrame(np.linalg.inv(I - B), index=B.index, columns=B.columns)

print(f"Inversa de Gosh:")
display(G.head())

Coeficientes de Gosh:


atividade,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,0.021056,0.012177,0.000492,0.000007,0.000018,0.000002,...,0.000167,0.000663,0.000667,0.000009,0.000714,0.0
0192,0.005447,0.032509,0.000941,0.000037,0.000108,0.000006,...,0.000106,0.000552,0.000833,0.000020,0.000252,0.0
0280,0.025755,0.030965,0.048358,0.000072,0.000036,0.000003,...,0.000131,0.000957,0.000410,0.000011,0.000085,0.0
0580,0.008062,0.017712,0.000588,0.015136,0.024983,0.000030,...,0.000066,0.000340,0.000304,0.000058,0.000147,0.0
0680,0.000037,0.000023,0.000002,0.000007,0.070423,0.000612,...,0.000058,0.000007,0.000027,0.000054,0.000077,0.0


Inversa de Gosh:


atividade,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,1.027732,0.024115,0.001430,0.000492,0.002646,0.000839,...,0.000747,0.003307,0.003419,0.000234,0.003882,0.0
0192,0.007038,1.040622,0.001315,0.000122,0.000593,0.000155,...,0.000887,0.004192,0.004288,0.000135,0.003702,0.0
0280,0.034144,0.038603,1.051213,0.000394,0.002122,0.000785,...,0.001312,0.003778,0.005376,0.000467,0.003539,0.0
0580,0.047379,0.030927,0.001452,1.016587,0.034717,0.002401,...,0.001661,0.006639,0.004758,0.000918,0.004615,0.0
0680,0.041332,0.015270,0.002051,0.003447,1.088226,0.010122,...,0.002695,0.004092,0.004790,0.001103,0.007397,0.0


## 4. Contabilidade de CO₂

Calcula as emissões atribuídas à produção e ao consumo por atividade.

Aplica as intensidades estimadas para 2015 à estrutura produtiva do mesmo ano; os resultados são expressos em Gg de CO₂.


### 4.1. Produção territorial

Calcula as emissões geradas por cada atividade no Brasil, incluindo a produção para exportação e excluindo as emissões ocorridas no exterior. A matriz `P` detalha essas emissões por atividade emissora e atividade do bem destinado à demanda final. Somar as colunas de cada linha recupera a emissão territorial da atividade, igual a `gamma * x`.


In [108]:
# Intensidades estimadas para 2015, em Gg de CO₂ por R$ milhão.
gamma = intensidades_co2[2015]

# Produção brasileira requerida pela demanda final total, em R$ milhões de 2015.
# y inclui a demanda doméstica e as exportações de produtos nacionais.
# Matriz 67 × 67: linhas = produtoras; colunas = atividades dos bens demandados.
producao_para_demanda_final = L.mul(y, axis=1)

# P - Emissões territoriais detalhadas por destino da produção.
# Aplica a intensidade brasileira às linhas da produção requerida.
# Matriz 67 × 67: linhas = atividades emissoras no Brasil;
# colunas = atividades dos bens destinados à demanda final.
# Unidade: Gg de CO₂; inclui as cadeias nacionais que atendem às exportações.
P = producao_para_demanda_final.mul(gamma, axis=0)

# Soma as colunas de cada linha, mantendo a atribuição à atividade emissora.
# Vetor com 67 atividades: emissões territoriais em Gg de CO₂.
producao_co2 = P.sum(axis=1)
assert (producao_co2 - gamma * x).abs().max() < 1e-8
print(f"Matrix de emissões por produção territorial:")
display(P.head())
print(f"Emissoes por atividade:")
display(producao_co2.head())


Matrix de emissões por produção territorial:


atividade,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,5860.351842,77.593758,8.452158,0.645615,9.441352,7.584493,...,7.077199,35.009390,32.806487,1.892582,35.022041,0.0
0192,22.962047,1915.937264,4.447070,0.091826,1.210539,0.800307,...,4.813287,25.390254,23.543854,0.627344,19.111755,0.0
0280,106.258225,67.790420,3391.515275,0.282351,4.132774,3.872142,...,6.786387,21.825844,28.154107,2.061877,17.423303,0.0
0580,38.063526,14.020151,1.209530,187.938579,17.450509,3.057408,...,2.218531,9.901431,6.431978,1.047042,5.866275,0.0
0680,38.223238,7.968226,1.966663,0.733458,629.656996,14.836133,...,4.143388,7.024812,7.453860,1.447993,10.823155,0.0


Emissoes por atividade:


atividade_origem
0191    10604.605714
0192     6067.940000
0280     5787.678571
0580     1494.070000
0680     1719.840000
dtype: float64

### 4.2. Consumo brasileiro

Calcula as emissões associadas à demanda final brasileira, somando três componentes:

- Emissões no Brasil para produzir os bens nacionais destinados à demanda final doméstica.
- Emissões no exterior para produzir os bens finais importados pelo Brasil.
- Emissões no exterior para produzir os insumos importados utilizados na produção brasileira destinada à demanda final doméstica.

As exportações já estão excluídas de `y_dom`, portanto suas emissões não entram na conta e não precisam ser subtraídas ao final.

In [109]:
# Intensidades estimadas para 2015, em Gg de CO₂ por R$ milhão.
gamma_ext = intensidades_co2_exterior[2015]
# Converte a demanda por produto em demanda por atividade produtora através da multiplicação por D.
# Vetores com 67 atividades cada, em R$ milhões de 2015.
y_dom = D @ y_dom_produtos
y_imp = D @ y_imp_produtos
y_exp = D @ y_exp_produtos
assert (y - y_dom - y_exp).abs().max() < 1e-6

# Converte Bm em A_imp: Insumos importados por atividade fornecedora e compradora.
# Matriz 67 × 67: linhas = atividades fornecedoras externas; colunas = compradoras brasileiras.
# Unidade: R$ de insumo importado por R$ de produção brasileira.
A_imp = D @ Bm
assert L_ext.index.equals(x.index) and L_ext.columns.equals(x.index)
assert A_imp.index.equals(x.index) and A_imp.columns.equals(x.index)

# Produção necessária em R$ milhões; cada resultado é uma matriz 67 × 67.
# Linhas: atividades produtoras; colunas: atividades dos bens da demanda final.
# Cada coluna de L é multiplicada pela demanda da atividade correspondente.
producao_nacional = L.mul(y_dom, axis=1)
producao_importacoes_finais = L_ext.mul(y_imp, axis=1)
producao_importacoes_intermediarias = L_ext @ A_imp @ producao_nacional

# Aplica a intensidade de emissão de cada atividade produtora às linhas.
# Resultados em Gg de CO₂, mantendo a origem e o destino das emissões separados.
# Cada resultado é uma matriz 67 × 67: linhas = atividades emissoras;
# colunas = atividades dos bens da demanda final.
C_nacional = producao_nacional.mul(gamma, axis=0)
C_final = producao_importacoes_finais.mul(gamma_ext, axis=0)
C_intermediaria = producao_importacoes_intermediarias.mul(gamma_ext, axis=0)

# Consumo brasileiro: produção nacional e importações; exportações ficam de fora.
# C: matriz 67 × 67; linhas = atividades emissoras; colunas = atividades dos bens demandados.
C = C_nacional + C_final + C_intermediaria
# consumo_co2: vetor com 67 atividades da demanda final, em Gg de CO₂.
consumo_co2 = C.sum(axis=0)


print(f"Matrix de emissões por consumo:")
display(C.head())
print(f"Emissoes por atividade:")
display(consumo_co2.head())



Matrix de emissões por consumo:


atividade,0191,0192,0280,0580,0680,0791,...,8592,8691,8692,9080,9480,9700
atividade_origem,,,,,,,,,,,,,
0191,2153.031855,73.950304,9.094972,0.267772,4.350501,0.348260,...,9.134061,43.073338,40.185581,2.633822,45.873160,0.0
0192,9.245007,1669.160013,4.847564,0.037720,0.604441,0.042089,...,5.558623,27.880167,25.458338,0.886883,25.669029,0.0
0280,47.906198,70.204795,3182.695665,0.124584,2.325668,0.224883,...,9.139288,28.946214,38.274850,2.905931,24.846290,0.0
0580,29.167514,17.376302,1.925253,61.226480,8.193601,0.247025,...,4.934604,16.439043,13.397130,2.298717,12.153675,0.0
0680,25.144631,11.688987,2.778467,0.340419,217.618413,0.788554,...,8.239206,13.139096,13.941513,2.744121,21.395784,0.0


Emissoes por atividade:


atividade
0191    5969.811678
0192    4161.232897
0280    3623.307695
0580     117.339110
0680    1834.917589
dtype: float64

## 5. Exportação das matrizes de emissões

Exporta `P` e `C` para `outputs/`, em CSV UTF-8, com 67 linhas e 67 colunas de valores em Gg de CO₂. A primeira coluna identifica a atividade emissora; os cabeçalhos identificam as atividades dos bens destinados à demanda final.

- `P`: emissões no Brasil associadas à demanda final total, incluindo exportações.
- `C`: emissões no Brasil e no exterior associadas à demanda final brasileira. As linhas agregam atividades equivalentes dos dois territórios.

Para a análise de redes, cada elemento representa uma atribuição de emissões diretas e indiretas à demanda final, não uma transação intermediária direta. Os valores da diagonal são preservados.

Para carregar em outro notebook, preservando os códigos como texto:

```python
P = pd.read_csv("outputs/matriz_emissoes_producao_2015.csv", dtype={"atividade_emissora": str}, index_col="atividade_emissora")
C = pd.read_csv("outputs/matriz_emissoes_consumo_2015.csv", dtype={"atividade_emissora": str}, index_col="atividade_emissora")
```

Os caminhos são relativos à raiz deste repositório.

O arquivo `setores_2015.csv` fornece os nomes dos códigos das atividades para a etapa de redes.


In [ ]:
# Exporta os valores calculados, sem arredondamento adicional ou totais.
# Linhas: atividades emissoras; colunas: atividades dos bens da demanda final.
pasta_outputs = dados.RAIZ_PROJETO / "outputs"
pasta_outputs.mkdir(exist_ok=True)

P.to_csv(pasta_outputs / "matriz_emissoes_producao_2015.csv", index_label="atividade_emissora", encoding="utf-8", lineterminator="\n")
C.to_csv(pasta_outputs / "matriz_emissoes_consumo_2015.csv", index_label="atividade_emissora", encoding="utf-8", lineterminator="\n")
print(f"Matrizes P e C exportadas para: {pasta_outputs}")

# Catálogo dos códigos exportados: informação descritiva para a etapa de redes.
setores = pd.DataFrame({"atividade": P.index, "descricao": tabelas["14"].iloc[5:72, 1].to_numpy()})
setores.to_csv(pasta_outputs / "setores_2015.csv", index=False, encoding="utf-8", lineterminator="\n")
